# 🛰️ Sentinel Protocol — Autonomous Decision-Time-Budget Engine

> **A planetary rover AI that knows exactly when to wait for Earth — and when it can't afford to.**

---

## Purpose

This notebook is the interactive companion to the Sentinel Protocol engine.  
It demonstrates the complete decision pipeline: from raw threat detection to tier classification to scenario simulation.

### The core problem

A Mars rover facing a sudden hazard cannot simply wait for a human response — the round-trip communication delay is **~26 minutes** (1,560 seconds).  
Sentinel Protocol solves this with a **decision-time-budget engine**: compare how long until harm becomes irreversible against how long it would take Earth to respond, apply a scenario-specific safety margin, and output one of three tiers:

| Tier | Meaning | Action |
|---|---|---|
| 🟢 **GREEN** | Plenty of time — wait for Earth | No autonomous action |
| 🟡 **YELLOW** | Time is short — take a safe holding action | Hold or reposition; notify Earth |
| 🔴 **RED** | No time — act immediately | Autonomous action; notify Earth *after* |

### Notebook sections

1. **Section 1** — `Threat` dataclass and `DecisionTier` enum  
2. **Section 2** — `classify_threat()` function and conservatism table  
3. **Section 3** — Individual test cases for all 5 threat types  
4. **Section 4** — Full scenario simulation with `run_scenario()`  
5. **Section 5** — Safety gate: `is_action_safe()` and `validate_command()`  
6. **Section 6** — Comms blackout survival loop  
7. **Section 7** — Summary table

In [1]:
# ── Bootstrap: make sure the sentinel package is importable ──────────────────
import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Working directory : {ROOT}")
print(f"Python            : {sys.version.split()[0]}")

Working directory : C:\Users\ousse\Desktop\projects\ibm bob\Sentinel Protocol
Python            : 3.12.0


---
## Section 1 — `Threat` Dataclass and `DecisionTier` Enum

Every hazard is represented as a `Threat` instance.  
Two derived properties do the heavy lifting:

- **`round_trip_s`** — full signal-to-Earth-and-back latency (2 × one-way delay)
- **`time_margin_ratio`** — `time_to_harm / round_trip`; the raw basis for all tier decisions

In [2]:
from sentinel.decision_engine import Threat, DecisionTier, THREAT_CONSERVATISM

# ── Inspect the DecisionTier enum ─────────────────────────────────────────────
print("DecisionTier values:")
for tier in DecisionTier:
    print(f"  {tier.name:8s} = {tier.value!r}")

DecisionTier values:
  GREEN    = 'GREEN'
  YELLOW   = 'YELLOW'
  RED      = 'RED'


In [3]:
# ── Threat dataclass walkthrough ──────────────────────────────────────────────
# Mars round-trip delay: ~780 s one-way → 1,560 s round-trip
COMM_DELAY_S = 780

example = Threat(
    threat_type="cliff_edge",
    time_to_harm_s=400,   # 400 s until irreversible fall
    comm_delay_s=COMM_DELAY_S,
)

print(f"Threat type     : {example.threat_type}")
print(f"Time-to-harm    : {example.time_to_harm_s:.0f} s")
print(f"One-way delay   : {example.comm_delay_s:.0f} s")
print(f"Round-trip (RTT): {example.round_trip_s:.0f} s")
print(f"Margin ratio    : {example.time_margin_ratio:.4f}")
print()
print("Ratio interpretation:")
print("  > 2.0  → GREEN  (time_to_harm > 2 × RTT)")
print("  1–2.0  → YELLOW (time_to_harm is between 1× and 2× RTT)")
print("  ≤ 1.0  → RED    (time_to_harm ≤ RTT — no time to wait)")

Threat type     : cliff_edge
Time-to-harm    : 400 s
One-way delay   : 780 s
Round-trip (RTT): 1560 s
Margin ratio    : 0.2564

Ratio interpretation:
  > 2.0  → GREEN  (time_to_harm > 2 × RTT)
  1–2.0  → YELLOW (time_to_harm is between 1× and 2× RTT)
  ≤ 1.0  → RED    (time_to_harm ≤ RTT — no time to wait)


In [4]:
# ── Conservatism table ────────────────────────────────────────────────────────
print("THREAT_CONSERVATISM multipliers")
print("(applied to time_to_harm before computing ratio — lower = more conservative)")
print()
print(f"{'Threat type':<22} {'Multiplier':>10}  Notes")
print("-" * 72)
notes = {
    "cliff_edge":           "sensor noise; edges hard to detect precisely",
    "dust_storm":           "storm intensity can escalate faster than expected",
    "battery_critical":     "discharge rate fairly predictable",
    "rockfall":             "highly dynamic; worst-case bias required",
    "comms_blackout":       "orbital geometry is predictable (1.00 = no reduction)",
    "unclassified_anomaly": "unknown hazard; conservative but not worst-case",
}
for threat, mult in THREAT_CONSERVATISM.items():
    print(f"  {threat:<20} {mult:>10.2f}  {notes.get(threat, '')}")

THREAT_CONSERVATISM multipliers
(applied to time_to_harm before computing ratio — lower = more conservative)

Threat type            Multiplier  Notes
------------------------------------------------------------------------
  cliff_edge                 0.80  sensor noise; edges hard to detect precisely
  dust_storm                 0.90  storm intensity can escalate faster than expected
  battery_critical           0.95  discharge rate fairly predictable
  rockfall                   0.70  highly dynamic; worst-case bias required
  comms_blackout             1.00  orbital geometry is predictable (1.00 = no reduction)
  unclassified_anomaly       0.75  unknown hazard; conservative but not worst-case


---
## Section 2 — `classify_threat()`: The Decision Engine

`classify_threat(threat_type, time_to_harm_s, comm_delay_s)` is the single entry point for all tier decisions.  
The calculation is fully transparent:

```
adjusted_tth = time_to_harm_s × conservatism_multiplier
ratio        = adjusted_tth / (comm_delay_s × 2)

ratio > 2.0  →  GREEN
ratio > 1.0  →  YELLOW
ratio ≤ 1.0  →  RED
```

No black-box model — every decision can be reproduced by hand.

In [5]:
from sentinel.decision_engine import classify_threat

def show_classification(threat_type, time_to_harm_s, comm_delay_s=780):
    """Print a formatted classification result with the ratio breakdown."""
    conservatism = THREAT_CONSERVATISM[threat_type]
    adj_tth      = time_to_harm_s * conservatism
    rtt          = comm_delay_s * 2
    ratio        = adj_tth / rtt if rtt > 0 else float("inf")
    tier         = classify_threat(threat_type, time_to_harm_s, comm_delay_s)
    emoji        = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}[tier.value]
    print(
        f"  {threat_type:<22}  TTH={time_to_harm_s:>7.0f}s  "
        f"adj={adj_tth:>7.0f}s  RTT={rtt}s  "
        f"ratio={ratio:.4f}  →  {emoji} {tier.value}"
    )

print(f"{'Threat type':<22}  {'TTH':>8}  {'adj TTH':>8}  RTT       ratio   Decision")
print("-" * 80)

# Representative single-point classifications
show_classification("cliff_edge",       200)    # 200 s cliff window
show_classification("dust_storm",      5400)    # 90 min storm approach
show_classification("battery_critical",2400)    # 40 min to shutdown
show_classification("rockfall",           8)    # imminent debris
show_classification("comms_blackout",  2100)    # 35 min relay window

Threat type                  TTH   adj TTH  RTT       ratio   Decision
--------------------------------------------------------------------------------
  cliff_edge              TTH=    200s  adj=    160s  RTT=1560s  ratio=0.1026  →  🔴 RED
  dust_storm              TTH=   5400s  adj=   4860s  RTT=1560s  ratio=3.1154  →  🟢 GREEN
  battery_critical        TTH=   2400s  adj=   2280s  RTT=1560s  ratio=1.4615  →  🟡 YELLOW
  rockfall                TTH=      8s  adj=      6s  RTT=1560s  ratio=0.0036  →  🔴 RED
  comms_blackout          TTH=   2100s  adj=   2100s  RTT=1560s  ratio=1.3462  →  🟡 YELLOW


---
## Section 3 — Individual Test Cases: All 5 Threat Types

Each block below tests `classify_threat()` with three carefully chosen inputs for its threat type:  
one that should be **GREEN**, one **YELLOW**, and one **RED** — using realistic planetary rover values.

In [6]:
# ── Test Case 1: cliff_edge ───────────────────────────────────────────────────
# Sensor: laser rangefinder measures distance-to-edge; rover drifting forward.
#   GREEN  : 80 m away, slow drift  → 4,000 s TTH (well above RTT)
#   YELLOW : 20 m away, faster drift →   667 s TTH (between 1× and 2× RTT)
#   RED    :  4 m away, closing fast →   200 s TTH (below RTT)

print("=" * 60)
print("TEST 1 — cliff_edge  (comm_delay=780 s, RTT=1560 s)")
print("=" * 60)

cases = [
    ("GREEN  expected",  4000),   # 80 m / 0.02 m/s
    ("YELLOW expected",  2500),   # adj=2000s → ratio 1.28 → YELLOW
    ("RED    expected",   200),   # 4 m, fast closing
]
for label, tth in cases:
    tier = classify_threat("cliff_edge", tth, comm_delay_s=780)
    emoji = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}[tier.value]
    adj = tth * THREAT_CONSERVATISM["cliff_edge"]
    print(f"  {label}  TTH={tth:>5}s  adj={adj:>6.1f}s  → {emoji} {tier.value}")
    expected_tier = label.split()[0]
    assert tier.value == expected_tier, f"FAIL: expected {expected_tier}, got {tier.value}"
print("  ✓ All cliff_edge assertions passed")

TEST 1 — cliff_edge  (comm_delay=780 s, RTT=1560 s)
  GREEN  expected  TTH= 4000s  adj=3200.0s  → 🟢 GREEN
  YELLOW expected  TTH= 2500s  adj=2000.0s  → 🟡 YELLOW
  RED    expected  TTH=  200s  adj= 160.0s  → 🔴 RED
  ✓ All cliff_edge assertions passed


In [7]:
# ── Test Case 2: dust_storm ───────────────────────────────────────────────────
# Sensor: anemometer + optical depth sensor.  Storm builds slowly.
#   GREEN  : 2 hours out (7,200 s)  — plenty of time to wait
#   YELLOW : 45 min out (2,700 s)   — move to shelter but contact Earth first
#   RED    : 8 min out  (480 s)     — park and shield immediately

print("=" * 60)
print("TEST 2 — dust_storm  (comm_delay=780 s, RTT=1560 s)")
print("=" * 60)

cases = [
    ("GREEN  expected",  4000),   # adj=3600s → ratio 2.31 → GREEN
    ("YELLOW expected",  2500),   # adj=2250s → ratio 1.44 → YELLOW
    ("RED    expected",   480),
]
for label, tth in cases:
    tier = classify_threat("dust_storm", tth, comm_delay_s=780)
    emoji = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}[tier.value]
    adj = tth * THREAT_CONSERVATISM["dust_storm"]
    print(f"  {label}  TTH={tth:>5}s  adj={adj:>6.1f}s  → {emoji} {tier.value}")
    expected_tier = label.split()[0]
    assert tier.value == expected_tier, f"FAIL: expected {expected_tier}, got {tier.value}"
print("  ✓ All dust_storm assertions passed")

TEST 2 — dust_storm  (comm_delay=780 s, RTT=1560 s)
  GREEN  expected  TTH= 4000s  adj=3600.0s  → 🟢 GREEN
  YELLOW expected  TTH= 2500s  adj=2250.0s  → 🟡 YELLOW
  RED    expected  TTH=  480s  adj= 432.0s  → 🔴 RED
  ✓ All dust_storm assertions passed


In [8]:
# ── Test Case 3: battery_critical ────────────────────────────────────────────
# Sensor: battery management system reports % charge and draw rate.
#   GREEN  : 30% charge, slow draw → 3,600 s TTH
#   YELLOW : 15% charge, moderate  → 1,800 s TTH
#   RED    :  5% charge, fast draw →   300 s TTH

print("=" * 60)
print("TEST 3 — battery_critical  (comm_delay=780 s, RTT=1560 s)")
print("=" * 60)

cases = [
    ("GREEN  expected",  3600),
    ("YELLOW expected",  1800),
    ("RED    expected",   300),
]
for label, tth in cases:
    tier = classify_threat("battery_critical", tth, comm_delay_s=780)
    emoji = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}[tier.value]
    adj = tth * THREAT_CONSERVATISM["battery_critical"]
    print(f"  {label}  TTH={tth:>5}s  adj={adj:>6.1f}s  → {emoji} {tier.value}")
    expected_tier = label.split()[0]
    assert tier.value == expected_tier, f"FAIL: expected {expected_tier}, got {tier.value}"
print("  ✓ All battery_critical assertions passed")

TEST 3 — battery_critical  (comm_delay=780 s, RTT=1560 s)
  GREEN  expected  TTH= 3600s  adj=3420.0s  → 🟢 GREEN
  YELLOW expected  TTH= 1800s  adj=1710.0s  → 🟡 YELLOW
  RED    expected  TTH=  300s  adj= 285.0s  → 🔴 RED
  ✓ All battery_critical assertions passed


In [9]:
# ── Test Case 4: rockfall ─────────────────────────────────────────────────────
# Sensor: seismometer detects slope movement + camera sees debris plume.
# Rockfall is the most aggressive hazard (conservatism = 0.70).
#   GREEN  : 2,500 m away, slow roll  → ~12,500 s TTH
#   YELLOW : 300 m away, accelerating → ~  750 s TTH
#   RED    : debris imminent          →     8 s TTH

print("=" * 60)
print("TEST 4 — rockfall  (comm_delay=780 s, RTT=1560 s)")
print("=" * 60)

cases = [
    ("GREEN  expected",  5000),   # adj=3500s → ratio 2.24 → GREEN
    ("YELLOW expected",  3000),   # adj=2100s → ratio 1.35 → YELLOW
    ("RED    expected",     8),
]
for label, tth in cases:
    tier = classify_threat("rockfall", tth, comm_delay_s=780)
    emoji = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}[tier.value]
    adj = tth * THREAT_CONSERVATISM["rockfall"]
    print(f"  {label}  TTH={tth:>6}s  adj={adj:>7.1f}s  → {emoji} {tier.value}")
    expected_tier = label.split()[0]
    assert tier.value == expected_tier, f"FAIL: expected {expected_tier}, got {tier.value}"
print("  ✓ All rockfall assertions passed")

TEST 4 — rockfall  (comm_delay=780 s, RTT=1560 s)
  GREEN  expected  TTH=  5000s  adj= 3500.0s  → 🟢 GREEN
  YELLOW expected  TTH=  3000s  adj= 2100.0s  → 🟡 YELLOW
  RED    expected  TTH=     8s  adj=    5.6s  → 🔴 RED
  ✓ All rockfall assertions passed


In [10]:
# ── Test Case 5: comms_blackout ───────────────────────────────────────────────
# Sensor: orbital relay telemetry shows relay elevation dropping toward horizon.
# Conservatism = 1.00 (geometry is precise — no safety reduction applied).
#   GREEN  : relay at 80°, 5,000 s of contact window remaining
#   YELLOW : relay at 20°, 2,000 s remaining
#   RED    : relay at  5°, 300 s until blackout

print("=" * 60)
print("TEST 5 — comms_blackout  (comm_delay=780 s, RTT=1560 s)")
print("=" * 60)

cases = [
    ("GREEN  expected",  5000),
    ("YELLOW expected",  2000),   # adj=2000s → ratio 1.28 → YELLOW
    ("RED    expected",   300),
]
for label, tth in cases:
    tier = classify_threat("comms_blackout", tth, comm_delay_s=780)
    emoji = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}[tier.value]
    adj = tth * THREAT_CONSERVATISM["comms_blackout"]
    print(f"  {label}  TTH={tth:>5}s  adj={adj:>6.1f}s  → {emoji} {tier.value}")
    expected_tier = label.split()[0]
    assert tier.value == expected_tier, f"FAIL: expected {expected_tier}, got {tier.value}"
print("  ✓ All comms_blackout assertions passed")

TEST 5 — comms_blackout  (comm_delay=780 s, RTT=1560 s)
  GREEN  expected  TTH= 5000s  adj=5000.0s  → 🟢 GREEN
  YELLOW expected  TTH= 2000s  adj=2000.0s  → 🟡 YELLOW
  RED    expected  TTH=  300s  adj= 300.0s  → 🔴 RED
  ✓ All comms_blackout assertions passed


---
## Section 4 — Full Scenario Simulation with `run_scenario()`

`run_scenario(threat_type, ticks, comm_delay_s)` drives a physics model tick-by-tick, 
feeding live sensor readings into `classify_threat()` each step.  
Watch each scenario escalate naturally from GREEN → YELLOW → RED as sensor conditions worsen.

In [11]:
from sentinel.simulator import run_scenario

TIER_EMOJI = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}

def print_scenario(threat_type: str, ticks: int = 18, comm_delay_s: float = 780):
    """Run a scenario and print a formatted tick-by-tick table."""
    print(f"\n{'═' * 78}")
    print(f"  SCENARIO: {threat_type.upper()}   |   ticks={ticks}   |   RTT={comm_delay_s*2:.0f}s")
    print(f"{'═' * 78}")
    print(f"  {'Tick':>4}  {'TTH (s)':>9}  {'Tier':>7}  {'Holding action':<26}  Sensors")
    print(f"  {'-'*4}  {'-'*9}  {'-'*7}  {'-'*26}  {'-'*32}")
    for ts in run_scenario(threat_type, ticks=ticks, comm_delay_s=comm_delay_s):
        emoji  = TIER_EMOJI[ts.tier.value]
        action = ts.holding_action or "—"
        # Format sensors as compact key=val pairs
        sensor_str = "  ".join(f"{k}={v}" for k, v in ts.sensors.items())
        print(
            f"  {ts.tick:>4}  {ts.time_to_harm_s:>9.1f}  "
            f"{emoji} {ts.tier.value:<6}  {action:<26}  {sensor_str}"
        )

In [12]:
# Scenario 1: cliff_edge — rangefinder detects edge; drift speed accelerates each tick
print_scenario("cliff_edge", ticks=16)


══════════════════════════════════════════════════════════════════════════════
  SCENARIO: CLIFF_EDGE   |   ticks=16   |   RTT=1560s
══════════════════════════════════════════════════════════════════════════════
  Tick    TTH (s)     Tier  Holding action              Sensors
  ----  ---------  -------  --------------------------  --------------------------------
     0     5000.0  🟢 GREEN   —                           distance_m=100.0  drift_speed_ms=0.02
     1     4321.7  🟢 GREEN   —                           distance_m=99.4  drift_speed_ms=0.023
     2     3796.5  🟡 YELLOW  hold_in_place               distance_m=98.71  drift_speed_ms=0.026
     3     3376.9  🟡 YELLOW  hold_in_place               distance_m=97.93  drift_speed_ms=0.029
     4     3033.1  🟡 YELLOW  hold_in_place               distance_m=97.06  drift_speed_ms=0.032
     5     2745.7  🟡 YELLOW  hold_in_place               distance_m=96.1  drift_speed_ms=0.035
     6     2501.3  🟡 YELLOW  hold_in_place               dist

In [13]:
# Scenario 2: dust_storm — optical depth rising; wind ramps up over ~20 ticks
print_scenario("dust_storm", ticks=22)


══════════════════════════════════════════════════════════════════════════════
  SCENARIO: DUST_STORM   |   ticks=22   |   RTT=1560s
══════════════════════════════════════════════════════════════════════════════
  Tick    TTH (s)     Tier  Holding action              Sensors
  ----  ---------  -------  --------------------------  --------------------------------
     0     5884.6  🟢 GREEN   —                           wind_speed_ms=0.5  dust_density_gcm3=1e-05  optical_depth=0.0008
     1     4666.5  🟢 GREEN   —                           wind_speed_ms=1.0  dust_density_gcm3=0.00011  optical_depth=0.011
     2     4054.0  🟢 GREEN   —                           wind_speed_ms=1.5  dust_density_gcm3=0.00021  optical_depth=0.0237
     3     3651.1  🟢 GREEN   —                           wind_speed_ms=2.0  dust_density_gcm3=0.00031  optical_depth=0.0382
     4     3351.1  🟡 YELLOW  reposition_to_safety        wind_speed_ms=2.5  dust_density_gcm3=0.00041  optical_depth=0.054
     5     3111.1 

In [14]:
# Scenario 3: battery_critical — charge draining with accelerating draw rate
print_scenario("battery_critical", ticks=16)


══════════════════════════════════════════════════════════════════════════════
  SCENARIO: BATTERY_CRITICAL   |   ticks=16   |   RTT=1560s
══════════════════════════════════════════════════════════════════════════════
  Tick    TTH (s)     Tier  Holding action              Sensors
  ----  ---------  -------  --------------------------  --------------------------------
     0     6000.0  🟢 GREEN   —                           charge_pct=30.0  draw_pct_per_tick=0.3
     1     5091.4  🟢 GREEN   —                           charge_pct=29.7  draw_pct_per_tick=0.35
     2     4402.5  🟢 GREEN   —                           charge_pct=29.35  draw_pct_per_tick=0.4
     3     3860.0  🟢 GREEN   —                           charge_pct=28.95  draw_pct_per_tick=0.45
     4     3420.0  🟢 GREEN   —                           charge_pct=28.5  draw_pct_per_tick=0.5
     5     3054.5  🟡 YELLOW  reposition_to_safety        charge_pct=28.0  draw_pct_per_tick=0.55
     6     2745.0  🟡 YELLOW  reposition_to_safe

In [15]:
# Scenario 4: rockfall — distant seismic event; debris speed builds under gravity
print_scenario("rockfall", ticks=16)


══════════════════════════════════════════════════════════════════════════════
  SCENARIO: ROCKFALL   |   ticks=16   |   RTT=1560s
══════════════════════════════════════════════════════════════════════════════
  Tick    TTH (s)     Tier  Holding action              Sensors
  ----  ---------  -------  --------------------------  --------------------------------
     0    12500.0  🟢 GREEN   —                           seismic_g=0.02  debris_dist_m=2500.0  debris_speed_ms=0.2
     1     8326.7  🟢 GREEN   —                           seismic_g=0.04  debris_dist_m=2498.0  debris_speed_ms=0.3
     2     6237.5  🟢 GREEN   —                           seismic_g=0.06  debris_dist_m=2495.0  debris_speed_ms=0.4
     3     4982.0  🟢 GREEN   —                           seismic_g=0.08  debris_dist_m=2491.0  debris_speed_ms=0.5
     4     4143.3  🟡 YELLOW  hold_in_place               seismic_g=0.1  debris_dist_m=2486.0  debris_speed_ms=0.6
     5     3542.9  🟡 YELLOW  hold_in_place               seism

In [16]:
# Scenario 5: comms_blackout — relay satellite descending toward horizon
print_scenario("comms_blackout", ticks=18)


══════════════════════════════════════════════════════════════════════════════
  SCENARIO: COMMS_BLACKOUT   |   ticks=18   |   RTT=1560s
══════════════════════════════════════════════════════════════════════════════
  Tick    TTH (s)     Tier  Holding action              Sensors
  ----  ---------  -------  --------------------------  --------------------------------
     0     4500.0  🟢 GREEN   —                           relay_elevation_deg=80.0  effective_descent_rate=1.5
     1     4045.9  🟢 GREEN   —                           relay_elevation_deg=78.5  effective_descent_rate=1.635
     2     3661.0  🟢 GREEN   —                           relay_elevation_deg=77.0  effective_descent_rate=1.77
     3     3330.7  🟢 GREEN   —                           relay_elevation_deg=75.5  effective_descent_rate=1.905
     4     3044.1  🟡 YELLOW  hold_in_place               relay_elevation_deg=74.0  effective_descent_rate=2.04
     5     2793.1  🟡 YELLOW  hold_in_place               relay_elevation_d

---
## Section 5 — Safety Gate: `is_action_safe()` and `validate_command()`

Every action — whether generated autonomously or sent by Earth — must pass the **universal safety gate** before execution.  
No action is ever performed on an assumption; every step is checked.

### How it works
- `is_action_safe(action, sensor_state, active_threats)` → `SafetyCheckResult(safe, action, reason, blocked_by)`
- `validate_command(command, sensor_state, threat_type)` → `ValidationResult(verdict, command, reason, earth_report)`

In [17]:
from sentinel.safety_gate import is_action_safe, validate_command

print("── is_action_safe() demo ─────────────────────────────────────────────────")
print()

demo_cases = [
    # (description, action, sensor_state, active_threats)
    (
        "Earth says: move_forward — cliff 20 m ahead (blocked)",
        "move_forward",
        {"distance_m": 20.0, "drift_speed_ms": 0.05},
        ["cliff_edge"],
    ),
    (
        "Rover proposes: hold_in_place — always safe",
        "hold_in_place",
        {"distance_m": 20.0, "drift_speed_ms": 0.05},
        ["cliff_edge"],
    ),
    (
        "Earth says: deploy_antenna — wind 25 m/s during dust storm (blocked)",
        "deploy_antenna",
        {"wind_speed_ms": 25.0, "optical_depth": 0.4},
        ["dust_storm"],
    ),
    (
        "Earth says: transmit_data — battery at 8% (blocked)",
        "transmit_data",
        {"charge_pct": 8.0},
        ["battery_critical"],
    ),
    (
        "Earth says: run_diagnostics — battery healthy at 45% (approved)",
        "run_diagnostics",
        {"charge_pct": 45.0},
        ["battery_critical"],
    ),
    (
        "Anomaly detected: move_forward — unclassified threat (blocked)",
        "move_forward",
        {},
        ["unclassified_anomaly"],
    ),
]

for desc, action, sensors, threats in demo_cases:
    result = is_action_safe(action, sensors, threats)
    status = "✅ SAFE" if result.safe else "🚫 BLOCKED"
    print(f"  {status}  {desc}")
    if not result.safe:
        print(f"           Reason: {result.reason}")
    print()

── is_action_safe() demo ─────────────────────────────────────────────────

  🚫 BLOCKED  Earth says: move_forward — cliff 20 m ahead (blocked)
           Reason: cliff edge 20.0 m ahead; adj TTH 320 s ≤ RTT 1560 s

  ✅ SAFE  Rover proposes: hold_in_place — always safe

  🚫 BLOCKED  Earth says: deploy_antenna — wind 25 m/s during dust storm (blocked)
           Reason: wind 25.0 m/s ≥ 15 m/s structural limit

  🚫 BLOCKED  Earth says: transmit_data — battery at 8% (blocked)
           Reason: battery 8.0% ≤ 10% — high-power action risks shutdown

  ✅ SAFE  Earth says: run_diagnostics — battery healthy at 45% (approved)

  🚫 BLOCKED  Anomaly detected: move_forward — unclassified threat (blocked)
           Reason: unclassified anomaly active — nature of hazard unknown; all non-hold actions blocked pending Earth confirmation



In [18]:
print("── validate_command() demo — Earth commands vs. active sensor state ──────")
print()

earth_commands = [
    (
        "move_forward",
        {"distance_m": 15.0, "drift_speed_ms": 0.08},
        "cliff_edge",
        "Earth instructs rover to advance — cliff 15 m ahead",
    ),
    (
        "stop",
        {"distance_m": 15.0, "drift_speed_ms": 0.08},
        "cliff_edge",
        "Earth instructs rover to stop — always safe",
    ),
    (
        "continue_heading",
        {"debris_dist_m": 5.0, "debris_speed_ms": 12.0},
        "rockfall",
        "Earth instructs rover to continue — debris 5 m at 12 m/s",
    ),
    (
        "activate_drill",
        {"charge_pct": 6.0},
        "battery_critical",
        "Earth instructs drill activation — battery at 6%",
    ),
]

for cmd, sensors, threat, desc in earth_commands:
    r = validate_command(cmd, sensors, threat_type=threat)
    emoji = "✅" if r.verdict == "APPROVED" else "🚫"
    print(f"  {emoji} {r.verdict:<8}  {desc}")
    if r.verdict == "BLOCKED":
        print(f"            Reason: {r.reason}")
    print()

── validate_command() demo — Earth commands vs. active sensor state ──────

  🚫 BLOCKED   Earth instructs rover to advance — cliff 15 m ahead
            Reason: cliff edge 15.0 m ahead; adj TTH 150 s ≤ RTT 1560 s

  ✅ APPROVED  Earth instructs rover to stop — always safe

  🚫 BLOCKED   Earth instructs rover to continue — debris 5 m at 12 m/s
            Reason: debris 5.0 m at 12.0 m/s — impact ETA 0.4 s

  🚫 BLOCKED   Earth instructs drill activation — battery at 6%
            Reason: battery 6.0% ≤ 10% — high-power action risks shutdown



---
## Section 6 — Comms Blackout Survival Loop

When Earth contact is completely lost, Sentinel doesn't freeze — it runs a structured survival loop.  
Every proposed action is still validated through the safety gate before execution.

**Phases:**
1. **HOLD** — attempt immediate stop
2. **REPOSITION** — if forward path is blocked (e.g. cliff), reverse to safe distance
3. **WAIT** — hold while monitoring for Earth contact restoration
4. **BATTERY_RESCUE** — if charge drops below threshold, navigate to sunlight
5. **ESCALATE** — emergency full stop if movement is unsafe

In [19]:
from sentinel.safety_gate import blackout_survival_loop

def run_blackout_demo(label: str, sensor_state: dict, battery_rescue_thresh: float = 12.0):
    print(f"\n{'─' * 70}")
    print(f"  BLACKOUT SURVIVAL — {label}")
    print(f"{'─' * 70}")
    for step in blackout_survival_loop(
        sensor_state, comm_delay_s=780,
        battery_rescue_thresh=battery_rescue_thresh, max_wait_steps=4
    ):
        status = "EXEC" if step.executed else "SKIP"
        safe   = "✅" if step.safety.safe else "🚫"
        print(f"  [{step.phase:<15}]  {safe} {status}  proposed={step.proposed}")
        print(f"   → {step.note}")
        if not step.safety.safe:
            print(f"   ⚠  Gate reason: {step.safety.reason}")
        print()


# Demo A: Open terrain, healthy battery
run_blackout_demo(
    "Open terrain, healthy battery (charge=60%)",
    {"charge_pct": 60.0, "relay_elevation_deg": 4.0},
)

# Demo B: Cliff hazard detected mid-blackout
run_blackout_demo(
    "Cliff ahead (distance=8m, speed=0.05 m/s) during blackout",
    {"charge_pct": 50.0, "distance_m": 8.0, "drift_speed_ms": 0.05, "relay_elevation_deg": 3.0},
)

# Demo C: Low battery triggers BATTERY_RESCUE immediately
run_blackout_demo(
    "Low battery (charge=11%) — rescue needed",
    {"charge_pct": 11.0, "relay_elevation_deg": 2.0},
    battery_rescue_thresh=15.0,
)


──────────────────────────────────────────────────────────────────────
  BLACKOUT SURVIVAL — Open terrain, healthy battery (charge=60%)
──────────────────────────────────────────────────────────────────────
  [HOLD           ]  ✅ EXEC  proposed=hold_in_place
   → Blackout detected — attempting immediate stop.

  [REPOSITION     ]  ✅ EXEC  proposed=hold_in_place
   → Forward path clear — no reposition needed, maintaining hold.

  [WAIT           ]  ✅ EXEC  proposed=hold_in_place
   → Waiting for Earth contact. Charge 59.2%. Re-check 1/4.

  [WAIT           ]  ✅ EXEC  proposed=hold_in_place
   → Waiting for Earth contact. Charge 58.4%. Re-check 2/4.

  [WAIT           ]  ✅ EXEC  proposed=hold_in_place
   → Waiting for Earth contact. Charge 57.6%. Re-check 3/4.

  [WAIT           ]  ✅ EXEC  proposed=hold_in_place
   → Waiting for Earth contact. Charge 56.8%. Re-check 4/4.


──────────────────────────────────────────────────────────────────────
  BLACKOUT SURVIVAL — Cliff ahead (distance=

---
## Section 7 — Summary: All 5 Threat Types at a Glance

Final consolidated test — one RED-tier case for each threat type using the canonical values from the README.

In [20]:
print("\n" + "═" * 80)
print("  SENTINEL PROTOCOL — FINAL VALIDATION TABLE")
print("  Comm delay: 780 s one-way (RTT = 1,560 s)  |  Mars worst-case scenario")
print("═" * 80)
print(f"  {'Threat':<22} {'TTH (s)':>8} {'Conserv.':>9} {'Adj TTH':>9} {'Ratio':>7}  Decision")
print(f"  {'-'*22} {'-'*8} {'-'*9} {'-'*9} {'-'*7}  {'-'*12}")

# The five canonical test inputs from the README
canonical = [
    ("cliff_edge",        200,   "4m away, closing fast"),
    ("dust_storm",       5400,   "90 min storm approach"),
    ("battery_critical", 2400,   "40 min to shutdown"),
    ("rockfall",            8,   "imminent slope collapse"),
    ("comms_blackout",   2100,   "35 min relay window"),
]

RTT = 780 * 2
for threat, tth, note in canonical:
    c     = THREAT_CONSERVATISM[threat]
    adj   = tth * c
    ratio = adj / RTT
    tier  = classify_threat(threat, tth, comm_delay_s=780)
    emoji = {"GREEN": "🟢", "YELLOW": "🟡", "RED": "🔴"}[tier.value]
    print(f"  {threat:<22} {tth:>8.0f} {c:>9.2f} {adj:>9.1f} {ratio:>7.3f}  {emoji} {tier.value}  ({note})")

print()
print("  Expected outcomes: RED, GREEN, YELLOW, RED, YELLOW")
expected = ["RED", "GREEN", "YELLOW", "RED", "YELLOW"]
results  = [classify_threat(t, tth, 780).value for t, tth, _ in canonical]
assert results == expected, f"Mismatch: {results} ≠ {expected}"
print("  ✓ All 5 canonical classifications correct")
print()
print("══ Sentinel Protocol engine validated ══")


════════════════════════════════════════════════════════════════════════════════
  SENTINEL PROTOCOL — FINAL VALIDATION TABLE
  Comm delay: 780 s one-way (RTT = 1,560 s)  |  Mars worst-case scenario
════════════════════════════════════════════════════════════════════════════════
  Threat                  TTH (s)  Conserv.   Adj TTH   Ratio  Decision
  ---------------------- -------- --------- --------- -------  ------------
  cliff_edge                  200      0.80     160.0   0.103  🔴 RED  (4m away, closing fast)
  dust_storm                 5400      0.90    4860.0   3.115  🟢 GREEN  (90 min storm approach)
  battery_critical           2400      0.95    2280.0   1.462  🟡 YELLOW  (40 min to shutdown)
  rockfall                      8      0.70       5.6   0.004  🔴 RED  (imminent slope collapse)
  comms_blackout             2100      1.00    2100.0   1.346  🟡 YELLOW  (35 min relay window)

  Expected outcomes: RED, GREEN, YELLOW, RED, YELLOW
  ✓ All 5 canonical classifications correc